# Sentiment Analysis — `fact_customer_reviews`

Reproduces the sentiment-scoring pipeline used to build `fact_customer_reviews_with_sentiment`
from the raw `fact_customer_reviews` table.

Uses **VADER** (Valence Aware Dictionary and sEntiment Reasoner) from NLTK to score review text,
then combines that score with each review's star **Rating** to flag cases where the written
tone and the numeric rating disagree (e.g. a 2-star review that reads positively).

**Output columns added:** `SentimentScore`, `SentimentCategory`, `SentimentBucket`


In [ ]:
import pandas as pd
import re
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Downloads the VADER lexicon (only needs to run once per environment)
nltk.download('vader_lexicon')

## 1. Load raw reviews

Replace the source below with wherever your `fact_customer_reviews` data actually lives.
In the original project this came from SQL Server (`PortfolioProject_MarketingAnalytics`),
queried before being handed to Python for sentiment scoring.


In [ ]:
# --- Option A: load from a CSV export of fact_customer_reviews ---
df = pd.read_csv('fact_customer_reviews.csv')

# --- Option B: pull directly from SQL Server instead ---
# import pyodbc
# conn = pyodbc.connect(
#     'DRIVER={ODBC Driver 17 for SQL Server};'
#     'SERVER=YOUR_SERVER\\SQLEXPRESS;'
#     'DATABASE=PortfolioProject_MarketingAnalytics;'
#     'Trusted_Connection=yes;'
# )
# df = pd.read_sql(
#     "SELECT ReviewID, CustomerID, ProductID, ReviewDate, Rating, ReviewText "
#     "FROM dbo.customer_reviews",
#     conn
# )

print(df.shape)
df.head()

## 2. Clean review text

Strips leading/trailing whitespace and collapses repeated internal spaces before scoring.


In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).strip()
    text = re.sub(r'\s+', ' ', text)  # collapse multiple spaces into one
    return text

df['ReviewText'] = df['ReviewText'].apply(clean_text)
df[['ReviewText']].head()

## 3. Score sentiment with VADER

VADER's `compound` score summarizes overall sentiment on a scale from **-1** (most negative)
to **+1** (most positive). This becomes `SentimentScore`.


In [ ]:
sia = SentimentIntensityAnalyzer()

def get_compound_score(text):
    return sia.polarity_scores(text)['compound']

df['SentimentScore'] = df['ReviewText'].apply(get_compound_score).round(4)
df[['ReviewText', 'SentimentScore']].head()

## 4. Classify `SentimentCategory`

Combines the customer's star **Rating** with the **sign** of the VADER score, so a rating that
disagrees with the text tone gets flagged as "Mixed" rather than just trusting one signal:

| Rating | Score sign | Category |
|---|---|---|
| 1–2 | ≤ 0 | Negative |
| 1–2 | > 0 | Mixed Negative *(low rating, positive-sounding text)* |
| 3 | < 0 | Mixed Negative |
| 3 | = 0 | Neutral |
| 3 | > 0 | Mixed Positive |
| 4–5 | any | Positive |


In [ ]:
def classify_sentiment(row):
    rating = row['Rating']
    score = row['SentimentScore']

    if rating <= 2:
        return 'Negative' if score <= 0 else 'Mixed Negative'
    elif rating == 3:
        if score < 0:
            return 'Mixed Negative'
        elif score == 0:
            return 'Neutral'
        else:
            return 'Mixed Positive'
    else:  # rating 4 or 5
        return 'Positive'

df['SentimentCategory'] = df.apply(classify_sentiment, axis=1)
df['SentimentCategory'].value_counts()

## 5. Bucket the raw score

Groups the continuous `SentimentScore` into four broad bands, useful for slicers/visuals in Power BI.


In [ ]:
def bucket_score(score):
    if score >= 0.5:
        return '0.5 to 1.0'
    elif score >= 0:
        return '0.0 to 0.49'
    elif score >= -0.5:
        return '-0.49 to 0.0'
    else:
        return '-1.0 to -0.5'

df['SentimentBucket'] = df['SentimentScore'].apply(bucket_score)
df['SentimentBucket'].value_counts()

## 6. Review the result


In [ ]:
print(df.shape)
df[['ReviewID', 'Rating', 'ReviewText', 'SentimentScore', 'SentimentCategory', 'SentimentBucket']].head(10)

## 7. Export

Saves the enriched table, matching the structure Power BI expects for `fact_customer_reviews_with_sentiment`.


In [ ]:
df.to_csv('fact_customer_reviews_with_sentiment.csv', index=False)
print('Saved fact_customer_reviews_with_sentiment.csv')

## 8. (Optional) Sanity-check against a known distribution

If you're reproducing the *original* portfolio dataset, `SentimentCategory` should come out
roughly as: Positive ≈ 840, Negative ≈ 226, Mixed Negative ≈ 196, Mixed Positive ≈ 86, Neutral ≈ 15.
Large deviations usually mean the input `ReviewText`/`Rating` differs from the original source data.
